# Experiment Analysis

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib widget

In [ ]:
# Needed to import from the enderscope library
# RUN ONLY ONCE

import os

os.chdir("..")

## Imports

In [ ]:
from pathlib import Path

from tqdm import tqdm
from rich.pretty import pprint

import numpy as np
import pandas as pd
import cv2

import panel as pn

import enderleaf.const as ec
from enderleaf.draw import image_grid, concat_tile_resize
from enderleaf.tools import read_dataframe, write_dataframe
from enderleaf.image import (
    load_image,
    to_pil,
    canny,
    find_circles,
    filter_circles,
    crop_image,
    Rectangle,
    merge_images,
    ImageMergeMode,
    match_previous_rotation,
    get_circles
)
from enderleaf.draw import draw_circles

In [ ]:
pn.extension()

## Constants

In [ ]:
EXP = "Exp00DM00"
INOC = "I0"
PLATE = 0
MONTH = 5
DAY = 19

PATH_TO_DATA = Path(".").joinpath("output", "job_data", EXP, INOC)
PATH_TO_IMAGES = Path(".").joinpath("output", "images", EXP, INOC)
MAX_CIRCLES=3

## Functions

In [ ]:
def load(row):
    return load_image(PATH_TO_IMAGES.joinpath(row.file_name))

## Load Data

In [ ]:
df = pd.concat([read_dataframe(f) for f in PATH_TO_DATA.glob("*.csv")]).sort_values(
    ["plate", "row", "col"]
).dropna(subset="north")
df = df[(df.plate == PLATE) & (df.month == MONTH) & (df.day == DAY)]
df["card_count"] = df[["north", "east", "west", "south"]].astype(int).sum(axis=1)
df["file_ok"] = df["file_name"].apply(lambda x:PATH_TO_IMAGES.joinpath(x).is_file())
df = df[df.file_ok == True]
df = df[df.job_ts != 20260515164717]
df["leaf_id"] = df.row.astype(str)+df.col.astype(str)
df

In [ ]:
pd.DataFrame(df.groupby(["job_ts", "card_count"]).height.mean()).reset_index()

## Select Cycle ID

In [ ]:
sel_leaf_id = pn.widgets.Select(
    name="col", options=list(df.leaf_id.unique()), sizing_mode="scale_width"
)
bt_random = pn.widgets.Button(name="Random Disc")

img_full = pn.pane.Image(sizing_mode="scale_width")
img_one = pn.pane.Image(sizing_mode="scale_width")
img_two = pn.pane.Image(sizing_mode="scale_width")


def on_random(event):
    row = df[["leaf_id"]].drop_duplicates().sample(n=1).iloc[0]
    sel_leaf_id.value = row.leaf_id


bt_random.on_click(on_random)


@pn.depends(*[w.param.value for w in [sel_leaf_id]], watch=True)
def on_ld_changed(leaf_id):
    df_ld = df[(df.leaf_id == leaf_id)]
    full_image = load(df_ld[df_ld.card_count == 4].reset_index(drop=True).iloc[0])
    height, width, _ = full_image.shape
    circles = get_circles(full_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    img_full.object = to_pil(crop_image(full_image, crop_data))
    img_one.object = to_pil(
        merge_images(
            image_list=[
                crop_image(load(row[1]), crop_data)
                for row in df_ld[df_ld.card_count == 1].iterrows()
            ],
            merge_mode=ImageMergeMode.MIN,
        )
    )
    # img_two.object = to_pil(
    #     merge_images(
    #         image_list=[
    #             crop_image(load(row[1]), crop_data)
    #             for row in df_ld[df_ld.card_count == 2].iterrows()
    #         ],
    #         merge_mode=ImageMergeMode.MIN,
    #     )
    # )


on_ld_changed(sel_leaf_id.value)

pn.Column(
    pn.Row(sel_leaf_id, bt_random),
    pn.Row(
        img_full,
        img_one,
        # img_two,
    ),
)

In [ ]:
sel_tile_leaf_id = pn.widgets.Select(
    name="col", options=list(df.leaf_id.unique()), sizing_mode="stretch_width"
)
sel_tile_lighting = pn.widgets.Select(
    name="lighting", options=[1, 2], sizing_mode="stretch_width"
)
bt_tile_random = pn.widgets.Button(name="Random Disc")

img_tile = pn.pane.Image(sizing_mode="stretch_width")


def on_random(event):
    row = df[["leaf_id"]].drop_duplicates().sample(n=1).iloc[0]
    sel_tile_leaf_id.value = row.leaf_id


bt_tile_random.on_click(on_random)


@pn.depends(*[w.param.value for w in [sel_tile_leaf_id, sel_tile_lighting]], watch=True)
def on_ld_changed(leaf_id, card_count):
    df_ld = df[(df.leaf_id == leaf_id) & (df.card_count == card_count)]
    full_image = load(df_ld.reset_index(drop=True).iloc[0])
    height, width, _ = full_image.shape
    circles = get_circles(full_image, color_space="hsv", channel="s", resize_factor=8)
    if len(circles["accepted"]) == 1:
        _, cx, cy, r = circles["accepted"][0]
        crop_data = Rectangle.from_circle((cx, cy, r + 16))
    else:
        crop_data = Rectangle(left=0, top=0, right=width, bottom=height)
    rows = [row[1] for row in df_ld.iterrows()]
    img_tile.object = to_pil(
        concat_tile_resize(
            [
                [crop_image(load(row), crop_data) for row in rows[:2]],
                [crop_image(load(row), crop_data) for row in rows[2:]],
            ]
        )
    )


on_ld_changed(sel_tile_leaf_id.value, sel_tile_lighting.value)

pn.Column(pn.Row(sel_tile_leaf_id, sel_tile_lighting, bt_tile_random), img_tile)

In [ ]:
df[
    (df.row == sel_row.value) 
    & (df.col == sel_col.value) 
    & (df.card_count == 1)
].iloc[0]

In [ ]:
sel_col.value

In [ ]:
df_id = df[df.cycle_id==cycle_id]
df_id

In [ ]:
accu, cx, cy, r = get_circles(load(df_id.iloc[0]), color_space="hsv", channel="s")[
    "accepted"
][0]
crop_data = Rectangle.from_circle((cx, cy, r + 16))

to_pil(
    concat_tile_resize(
        [
            [
                crop_image(load(df_id.iloc[0]), crop_data),
                merge_images(
                    image_list=[
                        crop_image(load(row[1]), crop_data) for row in df_id.iterrows()
                    ],
                    merge_mode=ImageMergeMode.MIN,
                ),
            ]
        ]
    )
)